In [1]:
import pandas as pd

def validate_lol_dataset(match_data_path, players_path, match_ids_path):
    print("--- Loading Datasets ---")
    try:
        df_matches = pd.read_csv(match_data_path)
        df_players = pd.read_csv(players_path)
        df_match_ids = pd.read_csv(match_ids_path)
        print("Successfully loaded all files.\n")
    except Exception as e:
        print(f"Error loading files: {e}")
        return

    # 1. Basic Shape and Structure Checks
    print("--- 1. Dataset Sizes ---")
    print(f"Match Data: {df_matches.shape[0]} rows, {df_matches.shape[1]} columns")
    print(f"Players: {df_players.shape[0]} rows, {df_players.shape[1]} columns")
    print(f"Match IDs: {df_match_ids.shape[0]} rows, {df_match_ids.shape[1]} columns\n")

    # 2. Check for Duplicate IDs
    print("--- 2. Uniqueness Checks ---")
    duplicate_matches = df_matches['matchId'].duplicated().sum()
    duplicate_players = df_players['puuid'].duplicated().sum()
    print(f"Duplicate matchIds in MatchData: {duplicate_matches}")
    print(f"Duplicate puuids in Players: {duplicate_players}\n")

    # 3. Cross-Reference Match IDs
    print("--- 3. Relational Integrity Checks ---")
    matches_in_data = set(df_matches['matchId'])
    matches_in_ref = set(df_match_ids['matchId'])
    
    missing_in_data = len(matches_in_ref - matches_in_data)
    missing_in_ref = len(matches_in_data - matches_in_ref)
    
    print(f"Match IDs in match_ids.csv but missing in MatchData.csv: {missing_in_data}")
    print(f"Match IDs in MatchData.csv but missing in match_ids.csv: {missing_in_ref}\n")

    # 4. Validate Flattened Participant Data
    print("--- 4. Participant Structure Check ---")
    # Verify that all 10 participants (0-9) have essential columns (e.g., ChampionName, Kills, Win)
    essential_suffixes = ['ChampionName', 'Kills', 'Deaths', 'Assists', 'Win']
    missing_columns = []
    
    for i in range(10):
        for suffix in essential_suffixes:
            col_name = f'participant{i}{suffix}'
            if col_name not in df_matches.columns:
                missing_columns.append(col_name)
                
    if missing_columns:
        print(f"WARNING: Missing expected participant columns: {missing_columns[:5]}... ({len(missing_columns)} total)")
    else:
        print("All 10 participants have the essential KDA, Champion, and Win columns.\n")

    # 5. Check Target Variables / Nulls
    print("--- 5. Target Variable & Missing Value Check ---")
    # Example: Check if gameDuration is valid (usually greater than 180 seconds for a real game)
    if 'gameDuration' in df_matches.columns:
        short_games = df_matches[df_matches['gameDuration'] < 180]
        print(f"Games shorter than 3 minutes (Remakes/Bugs): {len(short_games)}")

    # Quick check for heavy missing data across the match dataset
    null_counts = df_matches.isnull().sum()
    columns_with_nulls = null_counts[null_counts > 0]
    if not columns_with_nulls.empty:
        print(f"Columns with missing values found: {len(columns_with_nulls)} columns.")
        # Print top 5 columns with most missing values
        print(columns_with_nulls.sort_values(ascending=False).head())
    else:
        print("No missing values found in MatchData.csv.")

# --- Run the validation ---
# Replace these strings with the actual file paths if they are in a different directory
validate_lol_dataset('Data/Downloaded/matchData.csv', 'Data/Downloaded/players_8-14-25.csv', 'Data/Downloaded/match_ids.csv')

--- Loading Datasets ---
Successfully loaded all files.

--- 1. Dataset Sizes ---
Match Data: 101843 rows, 1770 columns
Players: 3560 rows, 9 columns
Match IDs: 101843 rows, 3 columns

--- 2. Uniqueness Checks ---
Duplicate matchIds in MatchData: 0
Duplicate puuids in Players: 0

--- 3. Relational Integrity Checks ---
Match IDs in match_ids.csv but missing in MatchData.csv: 0
Match IDs in MatchData.csv but missing in match_ids.csv: 0

--- 4. Participant Structure Check ---
All 10 participants have the essential KDA, Champion, and Win columns.

--- 5. Target Variable & Missing Value Check ---
Games shorter than 3 minutes (Remakes/Bugs): 1416
Columns with missing values found: 34 columns.
participant3SummonerName    101843
participant7SummonerName    101843
participant0SummonerName    101843
participant9SummonerName    101843
participant6SummonerName    101843
dtype: int64
